In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
learning_rate = 7e-4
max_iters = 50000
eval_iters = 1000

cuda


In [3]:
with open('books.txt', 'r', encoding = 'utf') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'à', 'æ', 'è', 'é', 'ê', 'ô', 'ù', '—', '‘', '’', '“', '”', '\ufeff']


In [4]:
string_to_int = { ch:i for i, ch in enumerate(chars)}
int_to_string = { i:ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda v: ''.join([int_to_string[c] for c in v])

enc = encode("Stulp")
print(decode(enc))


Stulp


In [5]:
data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([89, 24, 58, 51, 66, 70, 55, 68,  1, 30,  0, 41, 58, 55,  1, 24, 75, 53,
        62, 65, 64, 55,  0,  0,  0, 25, 65, 68, 65, 70, 58, 75,  1, 62, 59, 72,
        55, 54,  1, 59, 64,  1, 70, 58, 55,  1, 63, 59, 54, 69, 70,  1, 65, 56,
         1, 70, 58, 55,  1, 57, 68, 55, 51, 70,  1, 32, 51, 64, 69, 51, 69,  1,
        66, 68, 51, 59, 68, 59, 55, 69,  6,  1, 73, 59, 70, 58,  1, 42, 64, 53,
        62, 55,  0, 29, 55, 64, 68, 75,  6,  1])


In [6]:
n = int(0.8*len(data))
train_data = data[ :n]
valid_data = data[n: ]

def get_batch(split):
    data = train_data if split == "train" else valid_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

x, y = get_batch("train")
print("\n inputs:", x)
print("\n outputs:", y)



 inputs: tensor([[ 1, 44, 58, 55, 64,  1, 51,  1],
        [64, 54, 55, 68,  1, 70, 58, 51],
        [ 1, 75, 55, 70,  1, 70, 55, 68],
        [58, 55, 54, 57, 55, 69,  6, 88]], device='cuda:0')

 outputs: tensor([[44, 58, 55, 64,  1, 51,  1, 52],
        [54, 55, 68,  1, 70, 58, 51, 70],
        [75, 55, 70,  1, 70, 55, 68, 68],
        [55, 54, 57, 55, 69,  6, 88,  1]], device='cuda:0')


In [7]:
def evaluate_loss():
    out = {}
    model.eval()
    for i in ["train", "test"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(i)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[i] = losses.mean()
    model.train()
    return out
        

In [8]:
class BiGramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_emb_tb = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets = None):
        logits = self.token_emb_tb(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, index, max_new_tokens):

        for _ in range(max_new_tokens):

            logits, loss = self.forward(index)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim = -1)
            index_next = torch.multinomial(probs, num_samples = 1)
            index = torch.cat((index, index_next), dim = 1)

        return index

model = BiGramLM(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype = torch.long, device = device)
gen_chars = decode(m.generate(context, max_new_tokens = 500)[0].tolist())
print(gen_chars)


?meif[K﻿“C[QB2J1èu4v8-﻿;e.iIN﻿;P;;﻿h‘fmY‘pGJo,5ôtis[!kvf!sæ-0_àùZSxYk9-06 F06nMô]FXAJMB_v.zJe;Z
1eu?fA”.;uFr:EHiVEà:jà(eJ
ns]!8Weu3KKàùV6“AGGlZwO]n;N8eEéP﻿b!‘euk6me7êa[tWne*Y—f﻿Y:llæ:‘êIQ_40ôwb:ô)sB7l(q.ù_En‘M?Hi;2.5?dIàUU7!9Gx(kGt[ù]H’ETj-r﻿b!J:AGTj:7JéO:vpg05TF;11McrTx—(kk3êjSyG
ih*ôw8eWFlvzz?m6)ADOLP9[G]ps(g3rIR4:769v?(:U06ZéiUr-!:CU_æjuSéP;;:78zP_HO
vLùérV4(1:‘.LLh1J[KUTBæXlHp3’y0WFWàUHQ47oSp5:“IMô]uxRZpbVAQwAJ
moe﻿A?ul4s.Rq5BJéfLfTEY‘9U4o,ù]p sæQpP,XN”àZ.FEY﻿YR2nGQgéFblYF16d?f;WaKBy2àI.MJo:


In [9]:
"""x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is ", context, "print" , target)"""

'x = train_data[:block_size]\ny = train_data[1:block_size+1]\nfor t in range(block_size):\n    context = x[:t+1]\n    target = y[t]\n    print("when input is ", context, "print" , target)'

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):

    if iter % eval_iters == 0 :
        print(f"step: {iter}, {evaluate_loss()["train"]:.4f}, {evaluate_loss()["test"]:.4f}")
    xb, yb = get_batch("train")

    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()

print(loss.item())

step: 0, 2.4227, 2.4797
step: 1000, 2.4356, 2.4852
step: 2000, 2.4286, 2.4742
step: 3000, 2.4249, 2.4939
step: 4000, 2.4449, 2.4943
step: 5000, 2.4236, 2.4846
step: 6000, 2.4230, 2.4699
step: 7000, 2.4306, 2.4880
step: 8000, 2.4132, 2.4845
step: 9000, 2.4180, 2.4705
step: 10000, 2.4169, 2.4762
step: 11000, 2.4351, 2.4947
step: 12000, 2.4253, 2.4839
step: 13000, 2.4191, 2.4909
step: 14000, 2.4227, 2.4990
step: 15000, 2.4222, 2.4938
step: 16000, 2.4262, 2.4825
step: 17000, 2.4298, 2.4816
step: 18000, 2.4299, 2.4927
step: 19000, 2.4233, 2.4750
step: 20000, 2.4277, 2.4997
step: 21000, 2.4130, 2.4838
step: 22000, 2.4244, 2.5004
step: 23000, 2.4228, 2.4956
step: 24000, 2.4239, 2.4782
step: 25000, 2.4272, 2.4772
step: 26000, 2.4275, 2.4689
step: 27000, 2.4241, 2.4702
step: 28000, 2.4225, 2.4899
step: 29000, 2.4231, 2.4841
step: 30000, 2.4237, 2.4754
step: 31000, 2.4166, 2.4769
step: 32000, 2.4287, 2.4851
step: 33000, 2.4108, 2.4816
step: 34000, 2.4270, 2.4769
step: 35000, 2.4282, 2.4848
step:

In [14]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
gen_chars = decode(m.generate(context, max_new_tokens = 1000)[0].tolist())
print(gen_chars)


thay
Sur au kicithea I ng.
asenlakst bul Schis; And he lld s;[ld quprndel My pas s yollornd as w s il
athe t oospte abre d sounteatheeadd I wale llen lvectet theshely win mpood silldller plout sh t ny owhigle dar he y lank buthes mshe e
de.
Thivethy m antend wabuled We
f ounonagu s the heeare. Do tes ftrghanecowh ske I ure withis lthed Scy
quck
“Cht gan tuelie olarste th allthecowitid I
wa minge hoon mes y. whematimowe ea mithe bl, tomo ws be; aked hemmouterete topy se ied buf oony cell! huthes bore; me. ced stigere ngif my Ener t
forierersiowe
rdupas atsst on d ghe whicughinghasty ing
ts ns.

car! ceseit os
uis

ibis co  ftisat dmeretiethobu; de whe athefrmas aisfangrlkind wof plfachure

romallin, de wane I t s t anche imy, alpaved
I wowhoume an, nly oon crisorles toid aler
fecermibexindellor
st okimat bener, coureappr feabul wanca l wherund y hy s. Do hif Thir:‘Ciclin nkeras wh y, teer.
wnswit! bevid J‘Aunnd ghea
l biver; loud imet, te. misithe Ged, w. dmy inainoowaintons Y)06queari